# Week 3 — Wednesday: Choosing the Right Plot

**DATA 202 · Calvin University**

**Dataset:** same as Monday — homelessness data, now cleaned

- a table of numbers tells you little on its own
- the right chart reveals a pattern instantly; the wrong chart hides or distorts it just as fast

**Today's outline:**

- Quick clean, then load
- Matching a question to a plot type
- Histograms
- Scatter plots
- Line plots
- Bar charts
- Beyond these four — variation showcase
- When charts deceive

**Same cues as Monday:** 🎯 Predict First · 🙋 Quick Check

---
## Quick Clean, Then Load · ~2 min

Redo Monday's cleaning in one cell — this notebook stands on its own.

In [ ]:
import pandas as pd
import plotly.express as px

homeless = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/homeless.csv")

homeless["city"] = (
    homeless["city"].str.strip().str.replace("-", " ", regex=False)
    .str.replace(r"[^a-zA-Z\s]", "", regex=True).str.title()
    .replace({"Sf": "San Francisco", "La": "Los Angeles"})
)
homeless["shelter_status"] = (
    homeless["shelter_status"].str.strip().str.lower()
    .str.replace(r"^sheltered$", "shelter", regex=True)
    .str.replace(r"shelter\s*,\s*pending", "shelter pending", regex=True)
    .str.replace(r"temporary shelter", "shelter temporary", regex=True)
)
homeless["education_level"] = (
    homeless["education_level"].str.strip().str.lower()
    .replace({"none": "None", "primary": "Primary", "secondary": "Secondary", "higher": "Higher"})
)

homeless.head()

---
## Matching a Question to a Plot Type (SLO 03C) · ~3 min

| Question | Variable types | Plot |
|---|---|---|
| What's the distribution of one number? | one numeric | **Histogram** |
| How do two numbers relate to each other? | two numeric | **Scatter** |
| How does something change across an order? | numeric, ordered (time, rank, duration...) | **Line** |
| How do groups compare? | numeric + categorical | **Bar** |

Same idea every time: a column maps to a **visual channel** (x, y, color...) — the mapping decides what question the chart can answer.

🙋 **Quick Check:** compare *average* `monthly_support_usd` across the five cities. Which plot type fits, and why?

---
## Histograms · ~9 min

- divides a numeric variable into **bins**, counts rows per bin
- shows **center, spread, shape** — symmetric? skewed? multiple peaks? — and outliers
- **not** a bar chart: bar chart bars = categories; histogram bars = *ranges of a number*

🎯 **Predict First:** roughly symmetric, or skewed toward one side? Guess before we plot `monthly_support_usd`.

In [ ]:
px.histogram(homeless, x="monthly_support_usd", nbins=15,
             title="Distribution of Monthly Support ($)",
             labels={"monthly_support_usd": "Monthly Support (USD)"})

Try `nbins=5`, then `nbins=40`. What do you gain / lose at each? Whose interests might be served by a smoothed-out distribution vs. a spiky one — or the other way around?

---
### 🔨 Task 1 — Read a Histogram (~4 min)

Plot a histogram of `years_homeless`. Symmetric, or skewed? What would that shape mean for a program planning shelter capacity?

In [ ]:
# Your code here


---
## Scatter Plots · ~9 min

- shows the relationship between **two numeric variables** — one point per row
- good for trends, clusters, outliers — does one variable seem to predict another?

🎯 **Predict First:** `years_homeless` vs. `monthly_support_usd` — trend together, trend apart, or no clear relationship?

In [ ]:
px.scatter(homeless, x="years_homeless", y="monthly_support_usd", color="shelter_status",
           title="Years Homeless vs. Monthly Support",
           labels={"years_homeless": "Years Homeless", "monthly_support_usd": "Monthly Support (USD)"})

Was your prediction right? Does the pattern look different depending on `shelter_status`?

🙋 **Quick Check:** what does mapping `shelter_status` to `color=` add here that a plain black-and-white scatter couldn't show?

---
### 🔨 Task 2 — Build Your Own Scatter Plot (~4 min)

Plot `family_size` against `monthly_support_usd`, colored by `education_level`. Describe one pattern you see — or the lack of one.

In [ ]:
# Your code here


---
## Line Plots · ~8 min

- connects points **in order** — almost always time, but any meaningfully ordered variable works
- no dates here → order by `years_homeless` itself: as time homeless increases, how does *average* support change?
- one point per value of `years_homeless`, not one per person (Monday's skill feeds today's chart)

🎯 **Predict First:** rise, fall, or stay flat as years homeless increases?

In [ ]:
by_years = homeless.groupby("years_homeless")["monthly_support_usd"].mean().reset_index()

px.line(by_years, x="years_homeless", y="monthly_support_usd",
        title="Average Monthly Support by Years Homeless",
        labels={"years_homeless": "Years Homeless", "monthly_support_usd": "Average Monthly Support (USD)"})

🙋 **Quick Check:** this line jumps around instead of following a smooth trend. What does that tell you about how much data we have *per* value of `years_homeless`? (Hint: Monday's `groupby().count()`.)

---
### 🔨 Task 3 — Build Your Own Line Plot (~4 min)

Group by `years_homeless` again, plot the average `family_size`. Up, down, or flat as years homeless increases?

In [ ]:
# Your code here


---
## Bar Charts · ~8 min

- compares a number across categories
- vertical bars: a few categories; horizontal bars: many categories or long labels
- `color=` → turns one bar chart into a **grouped** or **stacked** comparison

🎯 **Predict First:** which `education_level` group gets the highest *average* monthly support? Guess before running.

In [ ]:
avg_support = homeless.groupby("education_level")["monthly_support_usd"].mean().reset_index()

px.bar(avg_support, x="education_level", y="monthly_support_usd",
       title="Average Monthly Support by Education Level",
       labels={"education_level": "Education Level", "monthly_support_usd": "Average Monthly Support (USD)"})

---
### 🔨 Task 4 — Build Your Own Bar Chart (~4 min)

Bar chart of the **count** of people per `city`, sorted most → fewest.

*Hint:* `groupby("city").size()` → `sort_values()` → `reset_index()` before plotting — or `orientation="h"` if the city labels get cramped.

In [ ]:
# Your code here


🙋 **Open question:** the line plot and both bar charts above all needed a `groupby()` first — the histogram and scatter plot earlier didn't. Why do line and bar charts need that collapsing step, when histograms and scatter plots don't?

---
## Beyond These Four — It's All Variations · ~5 min

Most other chart types are variations on these four, built by changing a channel (color, facet, orientation) rather than starting from scratch. A few, adapted to our homelessness data — same `homeless` DataFrame, no new dataset:

- **Bar** → flip it (horizontal), group it, stack it, or thin each bar to a single line (lollipop)
- **Line** → stack multiple lines and shade the area underneath (area chart)
- **Several numeric columns per category at once** → radar/spider plot

**First — a grouped bar chart**, a variation on the bar chart above: same `education_level` comparison, but with bars grouped by `shelter_status` instead of one bar per category.

In [ ]:
avg_support_grouped = homeless.groupby(["education_level", "shelter_status"])["monthly_support_usd"].mean().reset_index()

px.bar(avg_support_grouped, x="education_level", y="monthly_support_usd", color="shelter_status",
       barmode="group",
       title="Average Monthly Support by Education Level and Shelter Status",
       labels={"education_level": "Education Level", "monthly_support_usd": "Average Monthly Support (USD)"})

**Horizontal bar** — flip x and y, sort the bars so the pattern reads at a glance instead of forcing the eye to scan for the tallest.

In [ ]:
avg_support_by_city = homeless.groupby("city")["monthly_support_usd"].mean().reset_index()

fig_horizontal = px.bar(avg_support_by_city, x="monthly_support_usd", y="city",
                         orientation="h",
                         title="Average Monthly Support by City (Horizontal)",
                         labels={"city": "City", "monthly_support_usd": "Average Monthly Support (USD)"})
fig_horizontal.update_layout(yaxis={"categoryorder": "total ascending"})
fig_horizontal.show()

**Stacked bar** — instead of side-by-side bars, stack `shelter_status` counts on top of each other within each `education_level`: shows both the group total and its breakdown in one bar.

In [ ]:
status_counts = homeless.groupby(["education_level", "shelter_status"]).size().reset_index(name="count")

px.bar(status_counts, x="education_level", y="count", color="shelter_status",
       barmode="stack",
       title="Count of People by Education Level and Shelter Status (Stacked)",
       labels={"education_level": "Education Level", "count": "Count"})

**Lollipop** — a bar chart with the fill removed: a dot at the value, a thin line down to zero. Less "ink" for the same comparison.

In [ ]:
avg_years_by_city = homeless.groupby("city")["years_homeless"].mean().reset_index()

fig_lollipop = px.scatter(avg_years_by_city, x="city", y="years_homeless",
                           title="Average Years Homeless by City (Lollipop)",
                           labels={"city": "City", "years_homeless": "Average Years Homeless"})

for _, row in avg_years_by_city.iterrows():
    fig_lollipop.add_shape(type="line",
                            x0=row["city"], x1=row["city"],
                            y0=0, y1=row["years_homeless"],
                            line=dict(color="gray", width=2))

fig_lollipop.show()

**Area** — a line variation: stack `shelter_status` lines on top of each other across `years_homeless`, shaded underneath, so you can read both the trend and each group's share of the total at once.

In [ ]:
avg_support_by_year_status = homeless.groupby(["years_homeless", "shelter_status"])["monthly_support_usd"].mean().reset_index()

px.area(avg_support_by_year_status, x="years_homeless", y="monthly_support_usd", color="shelter_status",
        title="Average Monthly Support by Years Homeless and Shelter Status (Area)",
        labels={"years_homeless": "Years Homeless", "monthly_support_usd": "Average Monthly Support (USD)"})

**Radar/spider** — compares several numeric columns across categories at once. Each `education_level` gets one shape; each axis is a different (normalized) variable.

In [ ]:
cols = ["years_homeless", "monthly_support_usd", "family_size"]
avg_by_edu = homeless.groupby("education_level")[cols].mean().reset_index()
avg_by_edu[cols] = avg_by_edu[cols] / avg_by_edu[cols].max()  # normalize so axes are comparable

radar_long = avg_by_edu.melt(id_vars="education_level", value_vars=cols,
                              var_name="variable", value_name="value")

px.line_polar(radar_long, r="value", theta="variable", color="education_level",
              line_close=True,
              title="Education Level Compared Across Years Homeless, Support, and Family Size (Radar)")

---
## When Charts Deceive · ~4 min

Three broad ways charts mislead:

* **Misrepresentation** — the chart is simply wrong: cherry-picked data, distorted proportions, a truncated axis that exaggerates a difference
* **False impressions** — technically accurate, but a visual choice (3D effects, inconsistent colors, an unlabeled log scale) suggests a pattern that isn't really there
* **Ambiguity** — missing labels, units, or context leave the chart open to multiple readings

**Try it:** plot the *total* (not average) `monthly_support_usd` per `city`. Why might that number alone be misleading if you don't also show how many people are in each city?

In [ ]:
# Your code here — total monthly_support_usd per city


📎 **In class:** [Graphics Principles cheat sheet (the Novartis one)](graphics_principles.pdf) — we'll go through it together. Keep it handy for the Final Project.

**"Thumper Principle"** (from *Bambi*): if a chart can't say something useful, don't make it.

---
## Coming Up

| Day | Topic | Builds on today |
|---|---|---|
| Fri | Forum 1 — *Counting*, Ch. 1 | Every chart today made a choice about what to show — this week's reading asks who makes that choice, and for whom |
| Week 4 | Joining tables | Combining datasets *before* you can plot them together |
| Week 5 | Clustering | Finding groups the data suggests, instead of ones we chose (like `city` or `education_level`) in advance |